# PairwiseGP

この Notebook では、pairwise comparison から preference を学習する `robotorchan.models.PairwiseGP` を扱います。

通常の回帰と異なり、観測値はスカラー目的値ではなく「項目 i の方が項目 j より好ましい」といった比較結果です。

## 1. このモデルを使う場面

ユーザー・専門家・実験から絶対スコアより相対的な好みを自然かつ安定して取得できる場合に `PairwiseGP` を使います。人間の選好最適化、官能評価、ランキング、定量化しにくい材料スクリーニングなどが例です。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import PairwiseGP

torch.manual_seed(0)
dtype = torch.double

## 2. 潜在効用と比較データの生成

各候補点には直接観測しない潜在的な utility があるとし、その大小関係から比較データを生成します。`comparisons` の各行は `[winner, loser]` の順です。

In [ ]:
def latent_utility(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2.0 * torch.pi * x).squeeze(-1) + 0.35 * x.squeeze(-1)

n_items = 18
datapoints = torch.linspace(0.0, 1.0, n_items, dtype=dtype).unsqueeze(-1)
utility = latent_utility(datapoints)

n_comparisons = 50
pairs = torch.randint(0, n_items, (n_comparisons, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]

comparisons = []
for i, j in pairs.tolist():
    # PairwiseGP では各行を [winner, loser] として与えます。
    if utility[i] >= utility[j]:
        comparisons.append([i, j])
    else:
        comparisons.append([j, i])

comparisons = torch.tensor(comparisons, dtype=torch.long)
datapoints.shape, comparisons.shape

## 3. モデル構築と robotorchan API

通常の `train_X / train_Y` ではなく `datapoints / comparisons` を渡します。robotorchan は `raw_datapoints` と `raw_comparisons` を保持します。

In [ ]:
model = PairwiseGP(
    datapoints=datapoints,
    comparisons=comparisons,
)

print("raw_datapoints:", model.raw_datapoints.shape)
print("raw_comparisons:", model.raw_comparisons.shape)
print("supports_mll:", model.supports_mll)

mll = model.make_mll()
type(mll).__name__

## 4. Preference model の学習

`PairwiseGP` は `SingleTaskGP` の Gaussian regression likelihood とは異なり、Laplace approximation を利用します。`make_mll()` は `PairwiseLaplaceMarginalLogLikelihood` を返します。

In [ ]:
mll = model.make_mll()
fit_gpytorch_mll(mll)
model.eval()

## 5. 潜在 utility の posterior

比較データだけから学習したモデルに対し、連続グリッド上で潜在 utility の posterior を取得します。

In [ ]:
test_X = torch.linspace(0.0, 1.0, 250, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.sqrt().squeeze(-1)

# Pairwise utility は加法定数やスケールの規約を除いて識別されます。
# 可視化では両曲線を中心化して比較します。
true_u = latent_utility(test_X)
mean_centered = mean - mean.mean()
true_centered = true_u - true_u.mean()
lower = mean_centered - 1.96 * std
upper = mean_centered + 1.96 * std

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(test_X.squeeze(-1), true_centered, linestyle="--", label="true latent utility (centered)")
plt.plot(test_X.squeeze(-1), mean_centered, label="posterior mean (centered)")
plt.fill_between(test_X.squeeze(-1), lower, upper, alpha=0.2, label="95% interval")
plt.scatter(datapoints.squeeze(-1), utility - utility.mean(), s=25, alpha=0.5, label="items")
plt.xlabel("x")
plt.ylabel("relative utility")
plt.legend()
plt.title("PairwiseGP latent utility")
plt.show()

## 6. 現時点で好ましい領域を確認

学習には pairwise comparison しか使っていませんが、posterior mean を利用して候補を順位付けできます。ここでは posterior mean が最大となる位置を確認します。

In [ ]:
best_idx = mean.argmax()
best_x = test_X[best_idx]
print(f"posterior-best x: {best_x.item():.4f}")

## 7. このモデルを使わない方がよい場合

信頼できる絶対値のスカラー目的値を取得できるなら、`SingleTaskGP` などの通常回帰の方が単純で、1観測あたりの情報量も多いことが一般的です。`PairwiseGP` は測定・評価プロセス自体が比較形式である場合に特に有効です。